# Khối A: Chuẩn bị dữ liệu

**Đầu vào:** bộ dữ liệu Kaggle `khanfashee/nih-chest-x-ray-14-224x224-resized` (ảnh đã resize 224x224, file nhãn `Data_Entry_2017.csv`, danh sách test chính thức `test_list_NIH.txt`).

**Đầu ra:** `meta.csv`, mỗi dòng là một ảnh, gồm tên ảnh, mã bệnh nhân, ảnh thuộc `train_val` hay `test`, đường dẫn ảnh, và 14 cột nhãn 0/1.

**Nguồn dữ liệu:** bản resize do người dùng Kaggle khác thực hiện, không phải bản gốc của NIH.

## 1. Tải dữ liệu về Colab

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'nih-chest-x-ray-14-224x224-resized' dataset.
Path to dataset files: /kaggle/input/nih-chest-x-ray-14-224x224-resized


## 2. Tạo bảng meta.csv

Các bước: đọc file nhãn, tách thành 14 cột 0/1, đánh dấu ảnh test theo danh sách chính thức, gắn đường dẫn ảnh, lưu file.

Kết quả đúng: 112120 ảnh, 30805 bệnh nhân, 86524 train_val, 25596 test, ảnh 224x224, Hernia nhỏ nhất (0,0020).

In [5]:
import glob
import os

import pandas as pd
from PIL import Image

ROOT_DIR = "/kaggle/input/nih-chest-x-ray-14-224x224-resized"
IMAGE_DIR = ROOT_DIR + "/images-224"
LABEL_FILE = ROOT_DIR + "/Data_Entry_2017.csv"
TEST_FILE = ROOT_DIR + "/test_list_NIH.txt"
OUTPUT_FILE = "/content/meta.csv"

DISEASES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]

# Load metadata
df = pd.read_csv(LABEL_FILE)

# Create one binary column for each disease
for disease in DISEASES:
    df[disease] = df["Finding Labels"].apply(
        lambda s: int(disease in s.split("|"))
    )

# Assign official train/validation and test splits
test_images = set(open(TEST_FILE).read().split())

df["official"] = df["Image Index"].apply(
    lambda img: "test" if img in test_images else "train_val"
)

# Match image files by filename without extension
image_paths = {}

for ext in ("png", "jpg", "jpeg"):
    for path in glob.glob(
        IMAGE_DIR + "/**/*." + ext,
        recursive=True
    ):
        image_paths[
            os.path.splitext(os.path.basename(path))[0]
        ] = path

filename_no_ext = df["Image Index"].apply(
    lambda img: os.path.splitext(img)[0]
)

df["path"] = filename_no_ext.map(image_paths)

print(
    "Images found:",
    df["path"].notna().sum(),
    "out of",
    len(df),
    "metadata records"
)

# Remove entries without image files
df = df.dropna(subset=["path"])

# Rename columns
df = df.rename(
    columns={
        "Image Index": "image",
        "Patient ID": "patient"
    }
)

# Save metadata table
df[
    ["image", "patient", "official", "path"] + DISEASES
].to_csv(
    OUTPUT_FILE,
    index=False
)

# Dataset summary
print("Total images:", len(df))
print("Total patients:", df["patient"].nunique())

print(
    "Train/Validation images:",
    (df["official"] == "train_val").sum()
)

print(
    "Test images:",
    (df["official"] == "test").sum()
)

print(
    "Sample image size:",
    Image.open(df["path"].iloc[0]).size
)

print("Disease prevalence:")
print(df[DISEASES].mean().round(4))

Images found: 112120 out of 112120 metadata records
Total images: 112120
Total patients: 30805
Train/Validation images: 86524
Test images: 25596
Sample image size: (224, 224)
Disease prevalence:
Atelectasis           0.1029
Cardiomegaly          0.0247
Effusion              0.1187
Infiltration          0.1772
Mass                  0.0512
Nodule                0.0564
Pneumonia             0.0121
Pneumothorax          0.0473
Consolidation         0.0416
Edema                 0.0205
Emphysema             0.0224
Fibrosis              0.0150
Pleural_Thickening    0.0302
Hernia                0.0020
dtype: float64


## 3. Lưu meta.csv lên Google Drive

File trong `/content` bị xóa khi đóng phiên Colab, nên phải chép lên Drive.

In [6]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/CXR_Project/data
!cp /content/meta.csv /content/drive/MyDrive/CXR_Project/data/

Mounted at /content/drive


In [7]:
!ls -lh /content/drive/MyDrive/CXR_Project/data/

total 16M
-rw------- 1 root root 16M Oct  3 08:47 meta.csv
